# Scene exploration -- blf_scenes

`blf_scenes` パイプラインが生成したシーン (時間区間) を対話的に探索するノートブックです。
セグメンテーションが妥当か、ラベルが意図どおり付いているか、異常スコアの上位に何が来るかを
上から順に確認できます。

前提: `databricks bundle run blf_scenes` が完了し、対象カタログ / スキーマに
`blf_scenes`, `blf_scene_clusters`, `blf_scene_boundaries`, `blf_scene_signal_features`,
`blf_scene_feature_z`, `blf_scene_vectors` が存在すること。

ウィジェットを環境に合わせて設定してから、上から順に実行してください。


In [ ]:
# カタログ / スキーマと絞り込み条件を環境に合わせて変更してください。
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "blf_dev", "Schema")
dbutils.widgets.text("source_file", "", "Source file filter (部分一致、空なら全ファイル)")
dbutils.widgets.text("top_n", "20", "異常シーンの表示件数")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
SOURCE_FILE = dbutils.widgets.get("source_file").strip()
TOP_N = int(dbutils.widgets.get("top_n"))

if not CATALOG or not SCHEMA:
    raise ValueError("catalog and schema are required")

SCENES = f"`{CATALOG}`.`{SCHEMA}`.`blf_scenes`"
CLUSTERS = f"`{CATALOG}`.`{SCHEMA}`.`blf_scene_clusters`"
BOUNDARIES = f"`{CATALOG}`.`{SCHEMA}`.`blf_scene_boundaries`"
FEATURES = f"`{CATALOG}`.`{SCHEMA}`.`blf_scene_signal_features`"
FEATURE_Z = f"`{CATALOG}`.`{SCHEMA}`.`blf_scene_feature_z`"
VECTORS = f"`{CATALOG}`.`{SCHEMA}`.`blf_scene_vectors`"
GOLD = f"`{CATALOG}`.`{SCHEMA}`.`blf_gold_signals`"

# 空文字なら常に真になるので、ウィジェット未設定時は全ファイルが対象になります。
FILE_PRED = f"_source_file LIKE '%{SOURCE_FILE}%'" if SOURCE_FILE else "true"
print(f"scenes={SCENES}\nfilter={FILE_PRED}")

In [ ]:
"""共通のプロット設定。

Databricks ノートブックはライト / ダークどちらでも表示されるため、背景は透過にして
ノートブック側の地色を活かし、軸やグリッドは両方で読める中間グレーに固定します。
"""
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# クラスタ用の特徴量セット。feature_key の末尾はこのいずれかになります (§8 で名前を復元)。
CLUSTER_FEATURES = [
    "mean_v",
    "std_v",
    "delta_v",
    "range_v",
    "slope_per_s",
    "duty_cycle",
    "transition_rate_hz",
]

# 逐次 (magnitude) 用の単一色相ランプ。異常スコアや継続時間のような量に使います。
BLUE_SCALE = [
    [0.00, "#cde2fb"],
    [0.20, "#9ec5f4"],
    [0.40, "#6da7ec"],
    [0.60, "#3987e5"],
    [0.80, "#256abf"],
    [1.00, "#0d366b"],
]
BLUE = "#2a78d6"
# カテゴリ (identity) 用の固定順。順番に使い、巡回させないこと。
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK = "#8a8a85"
GRID = "rgba(138,138,133,0.25)"


def style(fig, title, xaxis_title=None, yaxis_title=None, height=420):
    """余計な装飾を落とし、軸とグリッドを控えめにした共通スタイルを当てます。"""
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color=INK)),
        height=height,
        margin=dict(l=10, r=10, t=50, b=10),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color=INK, size=12),
        legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0),
        hoverlabel=dict(font_size=12),
    )
    fig.update_xaxes(title=xaxis_title, gridcolor=GRID, zeroline=False, linecolor=GRID)
    fig.update_yaxes(title=yaxis_title, gridcolor=GRID, zeroline=False, linecolor=GRID)
    return fig

## 1. 全体サマリー

シーン数、対象ファイル数、ラベルが付いた割合、異常スコアの分布をまず確認します。
`labeled_pct` が極端に低い場合はルール CSV の signal_name が実データと合っていない可能性があります
(デモデータでは `ignition_status` のようにスネークケースの名前がある点に注意)。


In [ ]:
overview = spark.sql(f"""
    SELECT
        count(*)                                          AS n_scenes,
        count(DISTINCT _source_file)                      AS n_files,
        count(DISTINCT cluster_id)                        AS n_clusters,
        round(avg(duration_s), 2)                         AS mean_duration_s,
        round(100.0 * avg(CASE WHEN primary_action_label <> 'idle' THEN 1 ELSE 0 END), 1) AS labeled_pct,
        round(avg(anomaly_score), 1)                      AS mean_anomaly,
        round(percentile_approx(anomaly_score, 0.95), 1)  AS p95_anomaly,
        round(max(anomaly_score), 1)                      AS max_anomaly,
        round(avg(n_features_present), 1)                 AS mean_features_present
    FROM {SCENES}
    WHERE {FILE_PRED}
""")
display(overview)

## 2. セグメンテーションの健全性

シーンが細かく割れすぎていないか、ファイルを隙間なく覆えているかを確認します。

- `min_duration_s` が `scene_min_segment_seconds` を下回っていたら境界マージのバグ
- `gaps` / `overlaps` は 0 でなければならない (シーンはファイルを重複なく敷き詰める)
- `changepoint_pct` が 0 に近い場合は変化点が検出されていない (しきい値が高すぎる)


In [ ]:
per_file = spark.sql(f"""
    SELECT
        _source_file,
        count(*)                        AS n_scenes,
        round(min(duration_s), 3)       AS min_duration_s,
        round(avg(duration_s), 2)       AS mean_duration_s,
        round(max(duration_s), 2)       AS max_duration_s,
        round(min(seg_start_s), 2)      AS t_start_s,
        round(max(seg_end_s), 2)        AS t_end_s,
        round(100.0 * avg(CASE WHEN from_changepoint THEN 1 ELSE 0 END), 1) AS changepoint_pct
    FROM {BOUNDARIES}
    WHERE {FILE_PRED}
    GROUP BY _source_file
    ORDER BY _source_file
""")
display(per_file)

In [ ]:
# 隣接シーンの端点が一致しているかを lead() で突き合わせます。両方 0 が正常。
tiling = spark.sql(f"""
    WITH ordered AS (
        SELECT _source_file, seg_start_s, seg_end_s,
               lead(seg_start_s) OVER (PARTITION BY _source_file ORDER BY seg_start_s) AS next_start_s
        FROM {BOUNDARIES}
        WHERE {FILE_PRED}
    )
    SELECT
        sum(CASE WHEN next_start_s > seg_end_s + 1e-9 THEN 1 ELSE 0 END) AS gaps,
        sum(CASE WHEN next_start_s < seg_end_s - 1e-9 THEN 1 ELSE 0 END) AS overlaps
    FROM ordered
    WHERE next_start_s IS NOT NULL
""")
display(tiling)

In [ ]:
durations = spark.sql(f"""
    SELECT duration_s FROM {BOUNDARIES} WHERE {FILE_PRED}
""").toPandas()

fig = go.Figure(
    go.Histogram(
        x=durations["duration_s"],
        marker=dict(color=BLUE, line=dict(width=0)),
        nbinsx=40,
        hovertemplate="%{x:.1f} s<br>%{y} scenes<extra></extra>",
    )
)
style(fig, "シーン長の分布", "duration (s)", "scenes", height=340).show()

## 3. アクションラベルの分布

`primary_action_label` は優先度が最も高い (priority が最小の) 発火ルールです。
1 シーンに複数のラベルが付くこともあるため、`action_labels` を展開した内訳も併せて見ます。


In [ ]:
label_counts = spark.sql(f"""
    SELECT label, count(*) AS n_scenes
    FROM {SCENES} LATERAL VIEW explode(action_labels) t AS label
    WHERE {FILE_PRED}
    GROUP BY label
    ORDER BY n_scenes DESC
""").toPandas()

primary_counts = spark.sql(f"""
    SELECT primary_action_label, count(*) AS n_scenes,
           round(avg(anomaly_score), 1) AS mean_anomaly,
           round(avg(duration_s), 2)    AS mean_duration_s
    FROM {SCENES}
    WHERE {FILE_PRED}
    GROUP BY primary_action_label
    ORDER BY n_scenes DESC
""")
display(primary_counts)

In [ ]:
top_labels = label_counts.head(20).iloc[::-1]  # 横棒は下から積むので反転

fig = go.Figure(
    go.Bar(
        x=top_labels["n_scenes"],
        y=top_labels["label"],
        orientation="h",
        marker=dict(color=BLUE, line=dict(width=0)),
        text=top_labels["n_scenes"],
        textposition="outside",
        textfont=dict(color=INK),
        hovertemplate="%{y}<br>%{x} scenes<extra></extra>",
    )
)
style(fig, "付与されたアクションラベル (全件)", "scenes", None, height=max(340, 24 * len(top_labels) + 90))
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=False)
fig.show()

## 4. クラスタのプロファイル

`blf_scene_clusters` はクラスタごとの要約です。`action_labels` と `mean_speed_kmh` を見れば
そのクラスタが何の挙動なのかおおよそ読み取れます。

注意: `cluster_id` は同一のフィットに対しては安定 (サイズ降順で正規化済み) ですが、
入力データが変わる再実行をまたぐと durable な鍵ではありません。ダッシュボードなどは
`primary_action_label` 側に紐付けてください。


In [ ]:
display(
    spark.sql(f"""
    SELECT cluster_id, n_scenes, round(cluster_share, 4) AS cluster_share, cluster_name,
           round(mean_duration_s, 2) AS mean_duration_s,
           round(mean_speed_kmh, 1)  AS mean_speed_kmh,
           round(mean_anomaly_score, 1) AS mean_anomaly_score,
           round(median_distance, 3) AS median_distance,
           action_labels, k_used
    FROM {CLUSTERS}
    ORDER BY n_scenes DESC
""")
)

## 5. 異常シーンのランキング

`anomaly_score` は 0-100 で、セントロイド距離・ロバスト z の大きさ・クラスタの希少さを
合成したものです。`top_features` に符号付きの z が入っているので、なぜ高いのかを必ず
そこで確認してください。

`n_features_present` が極端に少ないシーンは、少数の特徴だけからスコアが出ているため
他のシーンと同列には比較できません。


In [ ]:
top_scenes = spark.sql(f"""
    SELECT _source_file, scene_id, segment_index,
           round(seg_start_s, 2) AS seg_start_s, round(seg_end_s, 2) AS seg_end_s,
           round(duration_s, 2)  AS duration_s,
           primary_action_label, cluster_id,
           round(anomaly_score, 1) AS anomaly_score,
           n_features_present,
           round(z_max, 2) AS z_max,
           transform(top_features, f -> concat(f.feature, '=', round(f.z, 1), 'z')) AS top_features,
           scene_summary
    FROM {SCENES}
    WHERE {FILE_PRED}
    ORDER BY anomaly_score DESC
    LIMIT {TOP_N}
""")
display(top_scenes)

In [ ]:
# 上位シーンで繰り返し効いている特徴を数えると、異常の «種類» が見えます。
display(
    spark.sql(f"""
    WITH ranked AS (
        SELECT scene_id, top_features
        FROM {SCENES}
        WHERE {FILE_PRED}
        ORDER BY anomaly_score DESC
        LIMIT {TOP_N}
    )
    SELECT f.feature, count(*) AS n_scenes, round(avg(abs(f.z)), 2) AS mean_abs_z
    FROM ranked LATERAL VIEW explode(top_features) t AS f
    GROUP BY f.feature
    ORDER BY n_scenes DESC, mean_abs_z DESC
""")
)

## 6. シーンのタイムライン

ファイルごとに時間軸上へシーンを並べ、異常スコアで濃淡を付けます。
色が濃い区間が «普段と違うことが起きていた» ところです。


In [ ]:
timeline = spark.sql(f"""
    SELECT _source_file, scene_id, seg_start_s, duration_s,
           anomaly_score, primary_action_label, cluster_id
    FROM {SCENES}
    WHERE {FILE_PRED}
    ORDER BY _source_file, seg_start_s
""").toPandas()

# フルパスは長いのでファイル名だけを軸ラベルにします。
timeline["file"] = timeline["_source_file"].str.rsplit("/", n=1).str[-1]

fig = go.Figure(
    go.Bar(
        x=timeline["duration_s"],
        base=timeline["seg_start_s"],
        y=timeline["file"],
        orientation="h",
        marker=dict(
            color=timeline["anomaly_score"],
            colorscale=BLUE_SCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(title=dict(text="anomaly", font=dict(color=INK)), tickfont=dict(color=INK)),
            # 隣接シーンが同じ濃さでも境目が分かるよう、細い区切り線を入れます。
            line=dict(width=0.5, color=INK),
        ),
        customdata=timeline[["primary_action_label", "cluster_id", "anomaly_score"]],
        hovertemplate=(
            "%{y}  t=%{base:.1f}-%{x:.1f}s<br>"
            "%{customdata[0]}<br>cluster %{customdata[1]} / anomaly %{customdata[2]:.0f}<extra></extra>"
        ),
    )
)
fig.update_layout(barmode="overlay", bargap=0.35)
style(
    fig,
    "シーンのタイムライン (濃いほど異常スコアが高い)",
    "time (s)",
    None,
    height=max(320, 46 * timeline["file"].nunique() + 120),
)
fig.update_yaxes(showgrid=False)
fig.show()

## 7. 特徴空間の俯瞰 (PCA)

`blf_scene_vectors` の密ベクトルを 2 次元に落として、シーンの散らばりを見ます。
クラスタ ID ではなく異常スコアで着色しているのは、8 色のカテゴリ配色を散布図
(全ペアが隣接しうる) に使うと色覚特性によっては見分けられないためです。
クラスタごとに見たいときは下のセルでファセット表示します。

固有値分解は numpy の SVD だけで行うので、追加の依存はありません。


In [ ]:
import numpy as np

vec_pdf = spark.sql(f"""
    SELECT v.scene_id, v.feature_vector, s.anomaly_score, s.cluster_id, s.primary_action_label
    FROM {VECTORS} v
    JOIN {SCENES} s USING (scene_id)
    WHERE {FILE_PRED}
""").toPandas()

if vec_pdf.empty:
    raise ValueError("blf_scene_vectors が空です。blf_scenes パイプラインの実行を確認してください")

X = np.vstack(vec_pdf["feature_vector"].to_numpy()).astype(float)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# パイプラインと同じロバストスケーリング。少数の外れ値に主成分を奪われないようにします。
med = np.nanmedian(X, axis=0)
mad = np.nanmedian(np.abs(X - med), axis=0)
Xs = (X - med) / np.where(mad > 0, 1.4826 * mad, 1.0)
Xs = np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)

Xc = Xs - Xs.mean(axis=0)
_, singular, vt = np.linalg.svd(Xc, full_matrices=False)
pcs = Xc @ vt[:2].T
explained = (singular**2 / (singular**2).sum())[:2]

vec_pdf["pc1"], vec_pdf["pc2"] = pcs[:, 0], pcs[:, 1]
print(f"{len(vec_pdf)} scenes, {X.shape[1]} features; PC1 {explained[0]:.1%} / PC2 {explained[1]:.1%} explained")

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=vec_pdf["pc1"],
        y=vec_pdf["pc2"],
        mode="markers",
        marker=dict(
            size=9,
            color=vec_pdf["anomaly_score"],
            colorscale=BLUE_SCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(title=dict(text="anomaly", font=dict(color=INK)), tickfont=dict(color=INK)),
            # 重なったマーカーを分離するための細いリング。
            line=dict(width=1, color="rgba(138,138,133,0.6)"),
        ),
        customdata=vec_pdf[["scene_id", "primary_action_label", "cluster_id"]],
        hovertemplate=("%{customdata[0]}<br>%{customdata[1]}<br>cluster %{customdata[2]}<extra></extra>"),
    )
)
style(fig, f"シーン特徴空間 (PC1 {explained[0]:.0%} / PC2 {explained[1]:.0%})", "PC1", "PC2", height=520).show()

In [ ]:
# クラスタごとのファセット。全体を薄いグレーで敷いた上に該当クラスタだけを重ねるので、
# どのクラスタが特徴空間のどこを占めるのかが一目で分かります。
cluster_ids = sorted(vec_pdf["cluster_id"].dropna().unique().tolist())
n_col = min(4, max(1, len(cluster_ids)))
n_row = (len(cluster_ids) + n_col - 1) // n_col

fig = make_subplots(
    rows=n_row,
    cols=n_col,
    subplot_titles=[f"cluster {int(c)}" for c in cluster_ids],
    shared_xaxes=True,
    shared_yaxes=True,
)

for i, cid in enumerate(cluster_ids):
    row, col = i // n_col + 1, i % n_col + 1
    fig.add_trace(
        go.Scatter(
            x=vec_pdf["pc1"],
            y=vec_pdf["pc2"],
            mode="markers",
            showlegend=False,
            marker=dict(size=5, color="rgba(138,138,133,0.22)"),
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )
    member = vec_pdf[vec_pdf["cluster_id"] == cid]
    fig.add_trace(
        go.Scatter(
            x=member["pc1"],
            y=member["pc2"],
            mode="markers",
            showlegend=False,
            marker=dict(size=7, color=BLUE, line=dict(width=0.5, color="rgba(138,138,133,0.6)")),
            customdata=member[["scene_id", "primary_action_label"]],
            hovertemplate="%{customdata[0]}<br>%{customdata[1]}<extra></extra>",
        ),
        row=row,
        col=col,
    )

style(fig, "クラスタ別の分布 (灰色は全シーン)", None, None, height=240 * n_row + 90)
fig.update_annotations(font=dict(color=INK, size=12))
fig.show()

## 8. 個別シーンのドリルダウン

最も異常スコアが高いシーンについて、寄与した特徴と元の生波形を並べて確認します。
`scene_id` を書き換えれば任意のシーンを見られます。


In [ ]:
# 既定では最上位の異常シーン。特定のシーンを見たいときは直接代入してください。
_top = spark.sql(f"""
    SELECT scene_id FROM {SCENES} WHERE {FILE_PRED} ORDER BY anomaly_score DESC LIMIT 1
""").collect()
if not _top:
    raise ValueError("該当するシーンがありません。source_file ウィジェットの条件を確認してください")
SCENE_ID = _top[0]["scene_id"]

# SCENE_ID = "dbfs:/Volumes/main/blf_dev/raw/drive001.blf#42"
print(SCENE_ID)

scene = spark.sql(f"""
    SELECT * FROM {SCENES} WHERE scene_id = '{SCENE_ID}'
""").collect()[0]

# f-string の式の中で辞書アクセスをせず、先に取り出しておきます
# (Databricks の Python 3.10/3.11 では入れ子の同種クォートが構文エラーになるため)。
SCENE_FILE = scene["_source_file"]
SCENE_START = float(scene["seg_start_s"])
SCENE_END = float(scene["seg_end_s"])

print(SCENE_FILE)
print(f"t = {SCENE_START:.2f} - {SCENE_END:.2f} s ({scene['duration_s']:.2f} s)")
print(
    f"label = {scene['primary_action_label']}  cluster = {scene['cluster_id']}  anomaly = {scene['anomaly_score']:.1f}"
)
if scene["scene_summary"]:
    print(f"summary: {scene['scene_summary']}")

In [ ]:
# このシーンで z の大きかった特徴。feature_key は 'signal_source|channel|signal_name.feature' 形式です。
display(
    spark.sql(f"""
    SELECT feature_key,
           round(feature_value, 4) AS feature_value,
           round(median_value, 4)  AS corpus_median,
           round(z, 2)             AS z
    FROM {FEATURE_Z}
    WHERE scene_id = '{SCENE_ID}'
    ORDER BY abs_z DESC
    LIMIT 15
""")
)

In [ ]:
# 寄与の大きかった信号の生波形を、シーン前後に余白を取って描きます。
PAD_S = 5.0

# feature_key は 'signal_source|channel|signal_name.feature' 形式。signal_name 自体に
# ドットを含む信号 (ETH の ip.ttl など) があるので、末尾の特徴量名だけを剥がします。
_SUFFIX = "|".join(CLUSTER_FEATURES)
top_signals = spark.sql(f"""
    SELECT split(feature_key, '[|]')[0] AS signal_source,
           regexp_replace(split(feature_key, '[|]')[2], '\\.({_SUFFIX})$', '') AS signal_name,
           max(abs_z) AS rank_z
    FROM {FEATURE_Z}
    WHERE scene_id = '{SCENE_ID}'
    GROUP BY 1, 2
    ORDER BY rank_z DESC
    LIMIT 4
""").toPandas()

if top_signals.empty:
    raise ValueError(f"no feature_z rows for {SCENE_ID}; blf_scene_feature_z を確認してください")

pred = " OR ".join(
    f"(signal_source = '{r.signal_source}' AND signal_name = '{r.signal_name}')" for r in top_signals.itertuples()
)
raw = spark.sql(f"""
    SELECT signal_source, signal_name, timestamp_s, signal_value
    FROM {GOLD}
    WHERE _source_file = '{SCENE_FILE}'
      AND timestamp_s BETWEEN {SCENE_START - PAD_S} AND {SCENE_END + PAD_S}
      AND ({pred})
    ORDER BY signal_source, signal_name, timestamp_s
""").toPandas()
print(f"{len(raw)} samples across {top_signals.shape[0]} signals")
display(top_signals)

In [ ]:
# 系列ごとに縦軸のスケールが違うので、二軸にはせず小さな倍数 (facet) で並べます。
# 単位の異なる信号を一つの図に重ねて二軸にするのは、比較を歪める典型的な失敗です。
series = list(raw.groupby(["signal_source", "signal_name"], sort=False))
if not series:
    raise ValueError("この区間に該当する生サンプルがありません (PAD_S を広げるか信号を確認してください)")

fig = make_subplots(
    rows=len(series),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[f"{src}  {name}" for (src, name), _ in series],
)

for i, ((src, name), part) in enumerate(series, start=1):
    fig.add_trace(
        go.Scatter(
            x=part["timestamp_s"],
            y=part["signal_value"],
            mode="lines",
            name=name,
            showlegend=False,
            line=dict(color=CATEGORICAL[(i - 1) % len(CATEGORICAL)], width=2),
            hovertemplate="t=%{x:.2f}s<br>%{y}<extra></extra>",
        ),
        row=i,
        col=1,
    )
    # シーンの範囲を陰影で示し、前後の余白と区別します。
    fig.add_vrect(x0=SCENE_START, x1=SCENE_END, fillcolor="rgba(42,120,214,0.10)", line_width=0, row=i, col=1)

style(fig, f"生波形 -- {SCENE_ID}", "time (s)", None, height=190 * len(series) + 110)
fig.update_annotations(font=dict(color=INK, size=12))
fig.show()

In [ ]:
# 同じシーンの全信号の集計値。ルールがなぜ発火した / しなかったのかはここで確認できます。
display(
    spark.sql(f"""
    SELECT signal_source, channel, signal_name, n_samples,
           round(mean_v, 3) AS mean_v, round(min_v, 3) AS min_v, round(max_v, 3) AS max_v,
           round(first_v, 3) AS first_v, round(last_v, 3) AS last_v,
           round(delta_v, 3) AS delta_v, round(range_v, 3) AS range_v,
           round(slope_per_s, 4) AS slope_per_s,
           n_transitions, round(duty_cycle, 3) AS duty_cycle
    FROM {FEATURES}
    WHERE scene_id = '{SCENE_ID}'
    ORDER BY signal_source, signal_name
""")
)

## 9. 調整の指針

探索の結果を踏まえてパイプラインを調整する場合、対応するのは次のパラメータです
(いずれも `databricks.yml` の変数、または DAB deploy 時の `--var` で変更)。

| 症状 | 見るべきパラメータ |
| --- | --- |
| シーンが細かすぎる / 多すぎる | `scene_window_seconds` を大きく、`scene_min_segment_seconds` を大きく |
| シーンが粗すぎて操作が埋もれる | `scene_window_seconds` を小さく、`scene_changepoint_k` を小さく |
| 変化点がほとんど検出されない | `scene_changepoint_k` を小さく、`scene_changepoint_signals` に対象信号を追加 |
| ラベルがほぼ `idle` | ルール CSV の `signal_name` が実データと一致しているか (§3 の分布と §8 の集計値を突き合わせる) |
| 異常スコアが全体的に高い / 低い | `scene_anomaly_weights`、`scene_anomaly_scale` |
| クラスタが偏る | `scene_k`、または `scene_baseline_scope` |

変更後は `databricks bundle deploy -t dev` してから `databricks bundle run blf_scenes` を実行し、
このノートブックを再実行して差分を確認してください。
